In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalogo", "catalog_dev")
catalogo = dbutils.widgets.get("catalogo").strip()
if catalogo not in ("catalog_dev", "catalog_prod"):
    raise ValueError("Catálogo no permitido")

def guardar_dimension(df, nombre):
    destino = f"{catalogo}.golden.{nombre}"
    columnas_destino = spark.table(destino).columns
    faltantes = set(columnas_destino) - set(df.columns)
    adicionales = set(df.columns) - set(columnas_destino)
    if faltantes or adicionales:
        raise ValueError(f"{nombre}: faltantes={faltantes}; adicionales={adicionales}")
    # insertInto inserta por posición; seleccionar en orden del destino es obligatorio.
    df.select(*columnas_destino).write.mode("overwrite").insertInto(destino)
    print(f"OK | {destino}: {spark.table(destino).count():,} registros")

print("Cargando dimensiones en:", catalogo)

In [0]:
fechas_pedidos = spark.table(f"{catalogo}.silver.pedidos").select(
    F.to_date("fecha_compra").alias("fecha")
)
fechas_resenas = spark.table(f"{catalogo}.silver.resenas").select(
    F.to_date("fecha_creacion").alias("fecha")
)
fechas_origen = fechas_pedidos.unionByName(fechas_resenas)
limites = fechas_origen.agg(
    F.min("fecha").alias("inicio"),
    F.max("fecha").alias("fin")
).first()

if limites["inicio"] is None or limites["fin"] is None:
    raise ValueError("No hay fechas válidas para construir dim_fecha")

meses = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio",
         "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"]
dias = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]

dim_fecha = (
    spark.range(1).select(
        F.explode(F.sequence(F.lit(limites["inicio"]), F.lit(limites["fin"]))).alias("fecha")
    )
    .withColumn("id_fecha", F.date_format("fecha", "yyyyMMdd").cast("int"))
    .withColumn("anio", F.year("fecha"))
    .withColumn("trimestre", F.quarter("fecha"))
    .withColumn("mes", F.month("fecha"))
    .withColumn("nombre_mes", F.element_at(F.array(*[F.lit(x) for x in meses]), F.col("mes")))
    .withColumn("semana_anio", F.weekofyear("fecha"))
    .withColumn("dia_mes", F.dayofmonth("fecha"))
    .withColumn("dia_semana", (F.pmod(F.dayofweek("fecha") + F.lit(5), F.lit(7)) + F.lit(1)).cast("int"))
    .withColumn("nombre_dia", F.element_at(F.array(*[F.lit(x) for x in dias]), F.col("dia_semana")))
)

guardar_dimension(dim_fecha, "dim_fecha")

In [0]:
dim_cliente = (
    spark.table(f"{catalogo}.silver.clientes")
    .select("id_cliente_unico")
    .filter(F.col("id_cliente_unico").isNotNull())
    .dropDuplicates(["id_cliente_unico"])
)
guardar_dimension(dim_cliente, "dim_cliente")

In [0]:
dim_producto = (
    spark.table(f"{catalogo}.silver.productos")
    .select("id_producto", "longitud_nombre", "longitud_descripcion", "cantidad_fotos",
            "peso_g", "largo_cm", "alto_cm", "ancho_cm")
    .filter(F.col("id_producto").isNotNull())
    .dropDuplicates(["id_producto"])
)
guardar_dimension(dim_producto, "dim_producto")

In [0]:
categorias_productos = (
    spark.table(f"{catalogo}.silver.productos")
    .select(F.col("categoria").alias("id_categoria"))
)
traducciones = spark.table(f"{catalogo}.silver.categorias").select(
    F.col("categoria").alias("id_categoria"), "categoria_ingles"
)

# Verificar unicidad antes del join para no multiplicar filas.
if traducciones.groupBy("id_categoria").count().filter(F.col("count") > 1).limit(1).count():
    raise ValueError("Existen categorías repetidas en Silver.categorias")

todas_categorias = (
    categorias_productos
    .unionByName(traducciones.select("id_categoria"))
    .filter(F.col("id_categoria").isNotNull())
    .dropDuplicates(["id_categoria"])
)

dim_categoria = (
    todas_categorias.alias("c")
    .join(traducciones.alias("t"), "id_categoria", "left")
    .select(
        F.col("id_categoria"),
        F.when(F.col("id_categoria") == "SIN CLASIFICAR", F.lit("SIN CLASIFICAR"))
         .otherwise(F.col("t.categoria_ingles")).alias("categoria_ingles")
    )
)
guardar_dimension(dim_categoria, "dim_categoria")

In [0]:
dim_vendedor = (
    spark.table(f"{catalogo}.silver.vendedores")
    .select("id_vendedor", "codigo_postal", "ciudad", "estado")
    .filter(F.col("id_vendedor").isNotNull())
    .dropDuplicates(["id_vendedor"])
)
guardar_dimension(dim_vendedor, "dim_vendedor")

In [0]:
dim_geografia = spark.table(f"{catalogo}.silver.ubicaciones").select(
    "codigo_postal", "ciudad", "estado", "latitud", "longitud", "registros_geograficos"
)
guardar_dimension(dim_geografia, "dim_geografia")

In [0]:
dim_metodo_pago = (
    spark.table(f"{catalogo}.silver.pagos")
    .select("metodo_pago")
    .filter(F.col("metodo_pago").isNotNull())
    .dropDuplicates(["metodo_pago"])
)
guardar_dimension(dim_metodo_pago, "dim_metodo_pago")

In [0]:
dim_estado_pedido = (
    spark.table(f"{catalogo}.silver.pedidos")
    .select("estado_pedido")
    .filter(F.col("estado_pedido").isNotNull())
    .dropDuplicates(["estado_pedido"])
)
guardar_dimension(dim_estado_pedido, "dim_estado_pedido")